# 使用 LlamaIndex 评估 RAG

在本 notebook 中，我们将研究如何使用 LlamaIndex 构建 RAG 流水线并对其进行评估。内容分为以下 3 个部分。

1. 理解检索增强生成（RAG）。
2. 使用 LlamaIndex 构建 RAG。
3. 使用 LlamaIndex 评估 RAG。

**检索增强生成（RAG）**

LLM 在海量数据集上训练，但这些数据集并不包含你的特定数据。检索增强生成（Retrieval-Augmented Generation，RAG）通过在生成过程中动态引入你的数据来解决这个问题。这并不是通过修改 LLM 的训练数据实现的，而是让模型能够实时访问和利用你的数据，从而提供更有针对性、更贴合上下文的响应。

在 RAG 中，你的数据会被加载并为查询做好准备，即“建立索引”。用户查询作用于索引，索引会将数据过滤为最相关的上下文。然后，这些上下文和你的查询会连同提示词一起发送给 LLM，由 LLM 给出响应。

即使你构建的是聊天机器人或智能体，也需要了解将数据引入应用的 RAG 技术。

![RAG 概览](../../images/llamaindex_rag_overview.png)

**RAG 的各个阶段**

RAG 中有五个关键阶段，它们也会成为你构建的任何大型应用的一部分。分别是：

**加载（Loading）：** 指将数据从其所在位置（无论是文本文件、PDF、其他网站、数据库还是 API）导入到你的流水线中。LlamaHub 提供了数百种连接器可供选择。

**索引（Indexing）：** 指创建一种便于查询数据的数据结构。对于 LLM 而言，这几乎总是意味着创建向量嵌入（即数据含义的数值表示），以及采用众多其他元数据策略，以便准确地找到与上下文相关的数据。

**存储（Storing）：** 数据建立索引后，你需要将索引连同其他元数据一起存储起来，以免需要重新建立索引。

**查询（Querying）：** 对于任何给定的索引策略，你都可以通过多种方式利用 LLM 和 LlamaIndex 数据结构进行查询，包括子查询、多步查询和混合策略。

**评估（Evaluation）：** 任何流水线中的关键一步，都是检查它相对于其他策略或在你做出更改时的有效性。评估可以客观衡量你对查询的响应有多准确、多忠实、多快速。

## 构建 RAG 系统。

既然我们已经理解了 RAG 系统的重要性，现在就来构建一个简单的 RAG 流水线。

In [ ]:
!pip install llama-index

In [1]:
# The nest_asyncio module enables the nesting of asynchronous functions within an already running async loop.
# This is necessary because Jupyter notebooks inherently operate in an asynchronous loop.
# By applying nest_asyncio, we can run additional async functions within this existing loop without conflicts.
import nest_asyncio

nest_asyncio.apply()

from llama_index.evaluation import generate_question_context_pairs
from llama_index import VectorStoreIndex, SimpleDirectoryReader, ServiceContext
from llama_index.node_parser import SimpleNodeParser
from llama_index.evaluation import generate_question_context_pairs
from llama_index.evaluation import RetrieverEvaluator
from llama_index.llms import OpenAI

import os
import pandas as pd

设置你的 OpenAI API 密钥

In [2]:
os.environ['OPENAI_API_KEY'] = 'YOUR OPENAI API KEY'

我们使用 [Paul Graham 的随笔文本](https://www.paulgraham.com/worked.html)来构建 RAG 流水线。

#### 下载数据

In [3]:
!mkdir -p 'data/paul_graham/'
!curl 'https://raw.githubusercontent.com/run-llama/llama_index/main/docs/examples/data/paul_graham/paul_graham_essay.txt' -o 'data/paul_graham/paul_graham_essay.txt'

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 75042  100 75042    0     0   190k      0 --:--:-- --:--:-- --:--:--  190k--:--  0:00:03 24586


#### 加载数据并构建索引。

In [4]:
documents = SimpleDirectoryReader("./data/paul_graham/").load_data()

# Define an LLM
llm = OpenAI(model="gpt-4")

# Build index with a chunk_size of 512
node_parser = SimpleNodeParser.from_defaults(chunk_size=512)
nodes = node_parser.get_nodes_from_documents(documents)
vector_index = VectorStoreIndex(nodes)

构建 QueryEngine 并开始查询。

In [5]:
query_engine = vector_index.as_query_engine()

In [6]:
response_vector = query_engine.query("What did the author do growing up?")

查看响应。

In [7]:
response_vector.response

'The author wrote short stories and worked on programming, specifically on an IBM 1401 computer using an early version of Fortran.'

默认情况下，它会检索 `two` 个相似的节点/文本块。你可以在 `vector_index.as_query_engine(similarity_top_k=k)` 中修改这一设置。

我们来查看这些检索到的节点中的文本。

In [8]:
# First retrieved node
response_vector.source_nodes[0].get_text()

'What I Worked On\n\nFebruary 2021\n\nBefore college the two main things I worked on, outside of school, were writing and programming. I didn\'t write essays. I wrote what beginning writers were supposed to write then, and probably still are: short stories. My stories were awful. They had hardly any plot, just characters with strong feelings, which I imagined made them deep.\n\nThe first programs I tried writing were on the IBM 1401 that our school district used for what was then called "data processing." This was in 9th grade, so I was 13 or 14. The school district\'s 1401 happened to be in the basement of our junior high school, and my friend Rich Draves and I got permission to use it. It was like a mini Bond villain\'s lair down there, with all these alien-looking machines — CPU, disk drives, printer, card reader — sitting up on a raised floor under bright fluorescent lights.\n\nThe language we used was an early version of Fortran. You had to type programs on punch cards, then stack

In [9]:
# Second retrieved node
response_vector.source_nodes[1].get_text()

"It felt like I was doing life right. I remember that because I was slightly dismayed at how novel it felt. The good news is that I had more moments like this over the next few years.\n\nIn the summer of 2016 we moved to England. We wanted our kids to see what it was like living in another country, and since I was a British citizen by birth, that seemed the obvious choice. We only meant to stay for a year, but we liked it so much that we still live there. So most of Bel was written in England.\n\nIn the fall of 2019, Bel was finally finished. Like McCarthy's original Lisp, it's a spec rather than an implementation, although like McCarthy's Lisp it's a spec expressed as code.\n\nNow that I could write essays again, I wrote a bunch about topics I'd had stacked up. I kept writing essays through 2020, but I also started to think about other things I could work on. How should I choose what to do? Well, how had I chosen what to work on in the past? I wrote an essay for myself to answer that 

我们已经构建了 RAG 流水线，现在需要评估它的性能。我们可以使用 LlamaIndex 的核心评估模块来评估 RAG 系统/查询引擎。下面来看看如何利用这些工具量化检索增强生成系统的质量。

## 评估

评估应作为衡量 RAG 应用的主要指标。它决定了流水线能否基于数据源和各种查询生成准确的响应。

虽然一开始检查单个查询和响应很有帮助，但随着边缘情况和失败案例的增多，这种方法可能变得不切实际。相反，建立一套汇总指标或自动化评估可能会更有效。这些工具可以帮助你洞察系统的整体性能，并指出可能需要更仔细审查的具体方面。

在 RAG 系统中，评估关注两个关键方面：

*   **检索评估：** 评估系统检索到的信息的准确性和相关性。
*   **响应评估：** 衡量系统基于检索到的信息生成的响应的质量和恰当性。

#### 问题-上下文对生成：

要评估 RAG 系统，必须有能够获取正确上下文、进而生成恰当响应的查询。`LlamaIndex` 提供了一个 `generate_question_context_pairs` 模块，专门用于构造问题与上下文对，可用于 RAG 系统的检索评估和响应评估。有关问题生成的更多详情，请参阅[文档](https://docs.llamaindex.ai/en/stable/examples/evaluation/QuestionGeneration.html)。

In [10]:
qa_dataset = generate_question_context_pairs(
    nodes,
    llm=llm,
    num_questions_per_chunk=2
)

100%|██████████| 58/58 [06:26<00:00,  6.67s/it]


#### 检索评估：

现在我们准备好进行检索评估了。我们将使用生成的评估数据集运行 `RetrieverEvaluator`。

我们首先创建 `Retriever`，然后定义两个函数：`get_eval_results` 用于在数据集上运行检索器，`display_results` 用于展示评估结果。

我们来创建检索器。

In [11]:
retriever = vector_index.as_retriever(similarity_top_k=2)

定义 `RetrieverEvaluator`。我们使用 **Hit Rate（命中率）** 和 **MRR** 指标来评估检索器。

**命中率：**

命中率计算的是在前 k 个检索文档中找到正确答案的查询所占的比例。简单来说，就是我们的系统在前几次猜测中猜对的频率。

**平均倒数排名（MRR）：**

对于每个查询，MRR 通过查看排名最高的相关文档的位次来评估系统的准确性。具体来说，它是所有查询中这些位次倒数的平均值。因此，如果第一个相关文档是排名第一的结果，倒数排名就是 1；如果是第二，倒数排名就是 1/2，依此类推。

我们来查看这些指标，检验检索器的性能。

In [12]:
retriever_evaluator = RetrieverEvaluator.from_metric_names(
    ["mrr", "hit_rate"], retriever=retriever
)

In [13]:
# Evaluate
eval_results = await retriever_evaluator.aevaluate_dataset(qa_dataset)

我们定义一个函数，以表格形式展示检索评估结果。

In [14]:
def display_results(name, eval_results):
    """Display results from evaluate."""

    metric_dicts = []
    for eval_result in eval_results:
        metric_dict = eval_result.metric_vals_dict
        metric_dicts.append(metric_dict)

    full_df = pd.DataFrame(metric_dicts)

    hit_rate = full_df["hit_rate"].mean()
    mrr = full_df["mrr"].mean()

    metric_df = pd.DataFrame(
        {"Retriever Name": [name], "Hit Rate": [hit_rate], "MRR": [mrr]}
    )

    return metric_df

In [15]:
display_results("OpenAI Embedding Retriever", eval_results)

,Retriever Name,Hit Rate,MRR
0,OpenAI Embedding Retriever,0.758621,0.62069


#### 观察：

使用 OpenAI Embedding 的检索器命中率为 `0.7586`，而 MRR 为 `0.6206`，这表明在确保最相关结果排在最前面方面仍有改进空间。MRR 低于命中率，说明排名靠前的结果并不总是最相关的。提升 MRR 可以考虑使用重排序器（reranker），对检索到的文档顺序进行优化。如需深入了解重排序器如何优化检索指标，请参阅我们[博客文章](https://blog.llamaindex.ai/boosting-rag-picking-the-best-embedding-reranker-models-42d079022e83)中的详细讨论。

#### 响应评估：

1. FaithfulnessEvaluator：衡量查询引擎的响应是否与某个源节点相符，可用于判断响应是否存在幻觉。
2. Relevancy Evaluator：衡量响应 + 源节点是否与查询相符。

In [16]:
# Get the list of queries from the above created dataset

queries = list(qa_dataset.queries.values())

#### 忠实度评估器

我们先从 FaithfulnessEvaluator 开始。

我们将使用 `gpt-3.5-turbo` 为给定查询生成响应，使用 `gpt-4` 进行评估。

我们分别为 `gpt-3.5-turbo` 和 `gpt-4` 创建 service_context。

In [17]:
# gpt-3.5-turbo
gpt35 = OpenAI(temperature=0, model="gpt-3.5-turbo")
service_context_gpt35 = ServiceContext.from_defaults(llm=gpt35)

# gpt-4
gpt4 = OpenAI(temperature=0, model="gpt-4")
service_context_gpt4 = ServiceContext.from_defaults(llm=gpt4)

使用 `gpt-3.5-turbo` 的 service_context 创建 `QueryEngine`，为查询生成响应。

In [18]:
vector_index = VectorStoreIndex(nodes, service_context = service_context_gpt35)
query_engine = vector_index.as_query_engine()

创建一个 FaithfulnessEvaluator。




In [19]:
from llama_index.evaluation import FaithfulnessEvaluator
faithfulness_gpt4 = FaithfulnessEvaluator(service_context=service_context_gpt4)

我们在一个问题上进行评估。

In [20]:
eval_query = queries[10]

eval_query

"Based on the author's experience and observations, why did he consider the AI practices during his first year of grad school as a hoax? Provide specific examples from the text to support your answer."

先生成响应，再使用忠实度评估器。

In [21]:
response_vector = query_engine.query(eval_query)

In [22]:
# Compute faithfulness evaluation

eval_result = faithfulness_gpt4.evaluate_response(response=response_vector)

In [23]:
# You can check passing parameter in eval_result if it passed the evaluation.
eval_result.passing

True

#### 相关性评估器

RelevancyEvaluator 可用于衡量响应和源节点（检索到的上下文）是否与查询相符，有助于判断响应是否真正回答了查询。

使用 `gpt-4` 实例化 `RelevancyEvaluator` 以进行相关性评估

In [24]:
from llama_index.evaluation import RelevancyEvaluator

relevancy_gpt4 = RelevancyEvaluator(service_context=service_context_gpt4)

我们对其中一个查询进行相关性评估。

In [25]:
# Pick a query
query = queries[10]

query

"Based on the author's experience and observations, why did he consider the AI practices during his first year of grad school as a hoax? Provide specific examples from the text to support your answer."

In [26]:
# Generate response.
# response_vector has response and source nodes (retrieved context)
response_vector = query_engine.query(query)

# Relevancy evaluation
eval_result = relevancy_gpt4.evaluate_response(
    query=query, response=response_vector
)

In [27]:
# You can check passing parameter in eval_result if it passed the evaluation.
eval_result.passing

True

In [28]:
# You can get the feedback for the evaluation.
eval_result.feedback

'YES'

#### 批量评估器：

我们已经分别完成了忠实度评估和相关性评估。LlamaIndex 提供了 `BatchEvalRunner`，可以批量计算多项评估。

In [29]:
from llama_index.evaluation import BatchEvalRunner

# Let's pick top 10 queries to do evaluation
batch_eval_queries = queries[:10]

# Initiate BatchEvalRunner to compute FaithFulness and Relevancy Evaluation.
runner = BatchEvalRunner(
    {"faithfulness": faithfulness_gpt4, "relevancy": relevancy_gpt4},
    workers=8,
)

# Compute evaluation
eval_results = await runner.aevaluate_queries(
    query_engine, queries=batch_eval_queries
)

In [30]:
# Let's get faithfulness score

faithfulness_score = sum(result.passing for result in eval_results['faithfulness']) / len(eval_results['faithfulness'])

faithfulness_score

1.0

In [31]:
# Let's get relevancy score

relevancy_score = sum(result.passing for result in eval_results['relevancy']) / len(eval_results['relevancy'])

relevancy_score


1.0

#### 观察：

忠实度得分为 `1.0`，表明生成的答案不含幻觉，完全基于检索到的上下文。

相关性得分为 `1.0`，表明生成的答案始终与检索到的上下文和查询保持一致。

## 结论

在本 notebook 中，我们探讨了如何使用 LlamaIndex 构建和评估 RAG 流水线，并重点评估了流水线中的检索系统和生成的响应。 

LlamaIndex 还提供了多种其他评估模块，你可以在[这里](https://docs.llamaindex.ai/en/stable/module_guides/evaluating/root.html)进一步了解